In [65]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [2]:
df1 = pd.read_excel(r"../india_weather_rainfall_data.xlsx")
df1.head()

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5


In [52]:
df2 = df1.copy()

In [53]:
df2["date_of_record"] = pd.to_datetime(df2["date_of_record"])
df2["day_of_year"] = df2["date_of_record"].dt.dayofyear
df2["year"] = df2["date_of_record"].dt.year
df2.head(5)

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall,day_of_year,year
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1,2,2021
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4,3,2021
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3,4,2021
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0,5,2021
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5,6,2021


In [54]:
df2.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp           43898
max_temp          110598
wind_speed        274444
air_pressure      304664
elevation              0
latitude               0
longitude              0
rainfall          257554
day_of_year            0
year                   0
dtype: int64

In [55]:
# Keep min_temp and max_temp for target prediction, but do not use them as input features.
# This also fixes notebook state if an earlier run already dropped these columns from df2.
for temp_col in ["min_temp", "max_temp"]:
    if temp_col not in df2.columns:
        df2[temp_col] = df1[temp_col]

df2.shape

(970339, 17)

In [56]:
for i, j in zip(df2.columns, df2.isnull().sum()):
    if j:
        print(f"{i}: {round(j/df2.shape[0]*100, 3)}%")

min_temp: 4.524%
max_temp: 11.398%
wind_speed: 28.283%
air_pressure: 31.398%
rainfall: 26.543%


In [57]:
df2["sin_day"] = np.sin(2 * np.pi * df2["day_of_year"] / 365.25)
df2["cos_day"] = np.cos(2 * np.pi * df2["day_of_year"] / 365.25)

# Keep rows with missing weather values; the model pipeline will impute them.
df2.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp           43898
max_temp          110598
wind_speed        274444
air_pressure      304664
elevation              0
latitude               0
longitude              0
rainfall          257554
day_of_year            0
year                   0
sin_day                0
cos_day                0
dtype: int64

In [58]:
df3 = df2.copy()

In [59]:
df3 = df3.sort_values("date_of_record")
cutoff_date = "2024-01-01"

train = df3[df3["date_of_record"] < cutoff_date]
test = df3[df3["date_of_record"] >= cutoff_date]

train.shape, test.shape

((805740, 19), (164599, 19))

In [60]:
feature_cols = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude", "month", "season", "state",
    "district", "station_name",
]
target_cols = ["avg_temp", "min_temp", "max_temp"]

X_train = train[feature_cols]
X_test = test[feature_cols]

X_train.shape, X_test.shape

((805740, 14), (164599, 14))

In [61]:
numeric_features = [
    "year", "sin_day", "cos_day", "rainfall", "wind_speed", "air_pressure",
    "elevation", "latitude", "longitude",
]
categorical_features = ["month", "season", "state", "district", "station_name"]

# Restore target columns if train/test were created before min_temp and max_temp were kept.
train = train.copy()
test = test.copy()
for temp_col in ["min_temp", "max_temp"]:
    if temp_col not in train.columns:
        train[temp_col] = df1.loc[train.index, temp_col]
    if temp_col not in test.columns:
        test[temp_col] = df1.loc[test.index, temp_col]


def build_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                HistGradientBoostingRegressor(
                    max_iter=220,
                    learning_rate=0.08,
                    l2_regularization=0.01,
                    random_state=42,
                ),
            ),
        ]
    )


models = {}
metrics = []

for target_col in target_cols:
    train_mask = train[target_col].notna()
    test_mask = test[target_col].notna()

    target_model = build_model()
    target_model.fit(train.loc[train_mask, feature_cols], train.loc[train_mask, target_col])

    predictions = target_model.predict(test.loc[test_mask, feature_cols])
    y_test = test.loc[test_mask, target_col]

    models[target_col] = target_model
    metrics.append(
        {
            "target": target_col,
            "train_rows": train_mask.sum(),
            "test_rows": test_mask.sum(),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

metrics_df = pd.DataFrame(metrics)
metrics_df

,target,train_rows,test_rows,MAE,RMSE,R2
0,avg_temp,805740,164599,1.294835,1.714200,0.913503
1,min_temp,761924,164517,1.414243,1.900571,0.910759
2,max_temp,695495,164246,1.659033,2.184245,0.855628


In [62]:
current_date = pd.Timestamp("2026-07-02")
current_day_of_year = current_date.dayofyear

kolkata_today = pd.DataFrame(
    [
        {
            "year": current_date.year,
            "sin_day": np.sin(2 * np.pi * current_day_of_year / 365.25),
            "cos_day": np.cos(2 * np.pi * current_day_of_year / 365.25),
            "rainfall": 5.2,       # mm, current daily precipitation estimate
            "wind_speed": 8.0,     # km/h
            "air_pressure": 999.0, # hPa / mb
            "elevation": 5,
            "latitude": 22.5333,
            "longitude": 88.3333,
            "month": "July",
            "season": "Monsoon",
            "state": "WB",
            "district": "Kolkata",
            "station_name": "Calcutta / Alipore",
        }
    ]
)

kolkata_predictions = {
    target_col: models[target_col].predict(kolkata_today[feature_cols])[0]
    for target_col in target_cols
}

print("Predicted temperatures for Kolkata today:")
print(f"Average: {kolkata_predictions['avg_temp']:.2f}°C")
print(f"Minimum: {kolkata_predictions['min_temp']:.2f}°C")
print(f"Maximum: {kolkata_predictions['max_temp']:.2f}°C")

pd.DataFrame([kolkata_predictions])

Predicted temperatures for Kolkata today:
Average: 29.28°C
Minimum: 26.28°C
Maximum: 33.28°C


,avg_temp,min_temp,max_temp
0,29.27837,26.283631,33.279357


In [63]:
from datetime import datetime, timedelta

yesterday = datetime.now() - timedelta(days=1)
day_of_year = yesterday.timetuple().tm_yday

kolkata_yesterday = pd.DataFrame([
    {
        "year": yesterday.year,
        "sin_day": np.sin(2 * np.pi * day_of_year / 365.25),
        "cos_day": np.cos(2 * np.pi * day_of_year / 365.25),

        # Approximate observed weather
        "rainfall": 12.4,          # mm
        "wind_speed": 8.2,         # km/h
        "air_pressure": 999.3,     # hPa

        # Static geographical features
        "elevation": 5,
        "latitude": 22.5333,
        "longitude": 88.3333,

        # Categorical features
        "month": yesterday.strftime("%B"),
        "season": "Monsoon",
        "state": "WB",
        "district": "Kolkata",
        "station_name": "Calcutta / Alipore",
    }
])

kolkata_yesterday_predictions = {
    target_col: models[target_col].predict(kolkata_yesterday[feature_cols])[0]
    for target_col in target_cols
}

print("Predicted temperatures for Kolkata yesterday:")
print(f"Average: {kolkata_yesterday_predictions['avg_temp']:.2f}°C")
print(f"Minimum: {kolkata_yesterday_predictions['min_temp']:.2f}°C")
print(f"Maximum: {kolkata_yesterday_predictions['max_temp']:.2f}°C")

pd.DataFrame([kolkata_yesterday_predictions])

Predicted temperatures for Kolkata yesterday:
Average: 28.84°C
Minimum: 26.09°C
Maximum: 33.20°C


,avg_temp,min_temp,max_temp
0,28.83639,26.085724,33.203896


In [66]:
def build_model_2():
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                CatBoostRegressor(
                    iterations=1000,
                    learning_rate=0.1,
                    depth=6,
                    loss_function='RMSE',
                    eval_metric='RMSE',
                    random_seed=42,
                    verbose=100
                ),
            ),
        ]
    )

models_2 = {}
metrics_2 = []

for target_col in target_cols:
    train_mask = train[target_col].notna()
    test_mask = test[target_col].notna()

    target_model = build_model_2()
    target_model.fit(train.loc[train_mask, feature_cols], train.loc[train_mask, target_col])

    predictions = target_model.predict(test.loc[test_mask, feature_cols])
    y_test = test.loc[test_mask, target_col]

    models_2[target_col] = target_model
    metrics_2.append(
        {
            "target": target_col,
            "train_rows": train_mask.sum(),
            "test_rows": test_mask.sum(),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        }
    )

metrics_2_df = pd.DataFrame(metrics_2)
metrics_2_df

0:	learn: 4.9708056	total: 81.3ms	remaining: 1m 21s
100:	learn: 1.6901101	total: 1.59s	remaining: 14.1s
200:	learn: 1.5592714	total: 3.1s	remaining: 12.3s
300:	learn: 1.4916399	total: 4.58s	remaining: 10.6s
400:	learn: 1.4418617	total: 6.05s	remaining: 9.04s
500:	learn: 1.4074774	total: 7.51s	remaining: 7.48s
600:	learn: 1.3798195	total: 8.98s	remaining: 5.96s
700:	learn: 1.3569744	total: 10.4s	remaining: 4.45s
800:	learn: 1.3367646	total: 11.9s	remaining: 2.95s
900:	learn: 1.3180510	total: 13.4s	remaining: 1.47s
999:	learn: 1.3031501	total: 14.8s	remaining: 0us
0:	learn: 5.4814644	total: 19ms	remaining: 19s
100:	learn: 1.7524435	total: 1.46s	remaining: 13s
200:	learn: 1.6308463	total: 2.95s	remaining: 11.7s
300:	learn: 1.5670324	total: 4.38s	remaining: 10.2s
400:	learn: 1.5236087	total: 5.8s	remaining: 8.66s
500:	learn: 1.4899455	total: 7.25s	remaining: 7.22s
600:	learn: 1.4645116	total: 8.62s	remaining: 5.72s
700:	learn: 1.4446389	total: 9.99s	remaining: 4.26s
800:	learn: 1.4277701	t

,target,train_rows,test_rows,MAE,RMSE,R2
0,avg_temp,805740,164599,1.294835,1.714200,0.913503
1,min_temp,761924,164517,1.414243,1.900571,0.910759
2,max_temp,695495,164246,1.659033,2.184245,0.855628


In [67]:
def predict_temperature_set(model_dict, input_row, model_name, case_name):
    predictions = {
        target_col: model_dict[target_col].predict(input_row[feature_cols])[0]
        for target_col in target_cols
    }

    return {
        "case": case_name,
        "model": model_name,
        "avg_temp": predictions["avg_temp"],
        "min_temp": predictions["min_temp"],
        "max_temp": predictions["max_temp"],
    }

comparison_rows = []
for case_name, input_row in [
    ("Kolkata today", kolkata_today),
    ("Kolkata yesterday", kolkata_yesterday),
]:
    comparison_rows.append(
        predict_temperature_set(models, input_row, "HistGradientBoosting", case_name)
    )
    comparison_rows.append(
        predict_temperature_set(models_2, input_row, "CatBoost", case_name)
    )

prediction_comparison = pd.DataFrame(comparison_rows)

comparison_diff = (
    prediction_comparison
    .pivot(index="case", columns="model", values=["avg_temp", "min_temp", "max_temp"])
)

for temp_col in target_cols:
    comparison_diff[(temp_col, "CatBoost - HistGradientBoosting")] = (
        comparison_diff[(temp_col, "CatBoost")]
        - comparison_diff[(temp_col, "HistGradientBoosting")]
    )

prediction_comparison.round(2), comparison_diff.round(2)

(                case                 model  avg_temp  min_temp  max_temp
 0      Kolkata today  HistGradientBoosting     29.28     26.28     33.28
 1      Kolkata today              CatBoost     29.81     26.84     33.34
 2  Kolkata yesterday  HistGradientBoosting     28.84     26.09     33.20
 3  Kolkata yesterday              CatBoost     29.22     26.64     32.79,
                   avg_temp                      min_temp                       \
 model             CatBoost HistGradientBoosting CatBoost HistGradientBoosting   
 case                                                                            
 Kolkata today        29.81                29.28    26.84                26.28   
 Kolkata yesterday    29.22                28.84    26.64                26.09   
 
                   max_temp                       \
 model             CatBoost HistGradientBoosting   
 case                                              
 Kolkata today        33.34                33.28   
 Kolkata 